In [1]:
# 1. Install dependencies
!pip install spacy pandas
!python -m spacy download en_core_web_sm

import pandas as pd
import json
import urllib.request
import spacy
import random
import os

# Create data directory
os.makedirs('/kaggle/working/data', exist_ok=True)

# Load Spacy NLP model
nlp = spacy.load("en_core_web_sm")

print("--- Step 1: Downloading RAW FOLIO Dataset ---")
url = "https://raw.githubusercontent.com/Yale-LILY/FOLIO/main/data/v0.0/folio-validation.jsonl"
raw_file_path = "/kaggle/working/data/folio_raw_original.jsonl"

urllib.request.urlretrieve(url, raw_file_path)
print(f"Raw dataset successfully saved to: {raw_file_path}")

data = []
with open(raw_file_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))
df = pd.DataFrame(data)

# Ensure premises are a single string
if isinstance(df['premises'].iloc[0], list):
    df['premises'] = df['premises'].apply(lambda x: ' '.join(x))
    
# Safely assign an ID
df['id'] = df.get('example_id', df.index)

print(f"Loaded {len(df)} samples into DataFrame.\n")

print("--- Step 2: Applying Caroline & Domain-Shift Filters (H1 Preparation) ---")

# 1. Caroline Filter (Nonsense Words - OOV tokens)
SYNTHETIC_NAMES = ["Boroogove", "Tweequ", "Snark", "Jubjub", "Bandersnatch", 
                   "Mome", "Rath", "Jabberwock", "Tumtum", "Gyre", "Vorpal"]

# 2. Domain-Shift Filter (Real Words, Unrelated Domain - In-Vocabulary tokens)
DOMAIN_SHIFT_NAMES = ["Electron", "Mitochondria", "Photon", "Galaxy", "Neutron", 
                      "Ribosome", "Quark", "Nebula", "Enzyme", "Chromosome", "Proton"]

def apply_filters(row):
    """
    Applies both Caroline and Domain-Shift filters consistently 
    across premises AND conclusions in the same row.
    """
    # Combine to find all unique entities in the whole context
    combined_text = row['premises'] + " " + row['conclusion']
    doc = nlp(combined_text)
    
    # Extract unique PERSON, ORG, GPE entities, sort by length (longest first)
    entities = sorted(list(set([ent.text for ent in doc.ents if ent.label_ in ['PERSON', 'ORG', 'GPE']])), key=len, reverse=True)
    
    # Setup pools
    pool_c = SYNTHETIC_NAMES.copy()
    pool_d = DOMAIN_SHIFT_NAMES.copy()
    random.shuffle(pool_c)
    random.shuffle(pool_d)
    
    # Create mappings
    map_c = {ent: pool_c.pop() for ent in entities if pool_c}
    map_d = {ent: pool_d.pop() for ent in entities if pool_d}
    
    # Apply replacements
    prem_c, conc_c = row['premises'], row['conclusion']
    prem_d, conc_d = row['premises'], row['conclusion']
    
    for ent in entities:
        if ent in map_c:
            prem_c = prem_c.replace(ent, map_c[ent])
            conc_c = conc_c.replace(ent, map_c[ent])
        if ent in map_d:
            prem_d = prem_d.replace(ent, map_d[ent])
            conc_d = conc_d.replace(ent, map_d[ent])
            
    return pd.Series([prem_c, conc_c, prem_d, conc_d])

# Apply the function
df[['caroline_premises', 'caroline_conclusion', 'domain_shift_premises', 'domain_shift_conclusion']] = df.apply(apply_filters, axis=1)

# Keep only necessary columns for the next step
df_clean = df[['id', 'premises', 'conclusion', 'label', 
               'caroline_premises', 'caroline_conclusion', 
               'domain_shift_premises', 'domain_shift_conclusion']]

processed_file_path = '/kaggle/working/data/folio_h1_experiment_v2.csv'
df_clean.to_csv(processed_file_path, index=False)

print(f"Processed dataset successfully saved to: {processed_file_path}")
print("\n--- Let's look at a Domain-Shift example ---")
sample = df_clean.iloc[9] # The Wild Turkey example
print(f"Original Premise snippet: {sample['premises'][:80]}...")
print(f"Caroline Premise snippet: {sample['caroline_premises'][:80]}...")
print(f"Domain-Shift Premise snippet: {sample['domain_shift_premises'][:80]}...")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 56.1 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
--- Step 1: Downloading RAW FOLIO Dataset ---
Raw dataset successfully saved to: /kaggle/working/data/folio_raw_original.jsonl
Loaded 204 samples into DataFrame.

--- Step 2: Applying Caroline & Domain-Shift Filters (H1 Preparation) ---
Processed dataset successfully saved to: /kaggle/working/data/folio_h1_experiment_v2.csv

--- Let's look at a Domain-Shift example ---
Original Premise snippet: There are six types of wild turkeys: Eastern wild turkey, Osceola wild turkey, G...
Caroline Premise snippet: There are six types of wild turkeys: Vorpal wild turkey, Jubjub wi